In [0]:

from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col, explode_outer,from_unixtime
from delta.tables import DeltaTable




##############################
#######--SALES ORDERS--#######
##############################

def sales_orders(df):
    sales_orders = df.select("order_number",
                            "customer_id",
                            "customer_name",
                            "number_of_line_items",
                            from_unixtime(col("order_datetime")).alias("order_timestamps")).distinct()
    return sales_orders

##############################
######--ORDERD PRODUCTS--#####
##############################

def orderd_product(df):
        orderd_product_schema = ArrayType(
            StructType([
                StructField("curr", StringType()),
                StructField("id", StringType()),
                StructField("name", StringType()),
                StructField("price", StringType()),
                StructField("promotion_info", StringType()),
                StructField("qty", StringType()),
                StructField("unit", StringType())]
            ))

        df_parsed = df.withColumn("ordered_products", from_json(col("ordered_products"), orderd_product_schema))\
                    .withColumn("ordered_products",explode_outer("ordered_products"))\
                    .select(
                        "customer_id",
                        "customer_name",
                        "order_number",
                        "ordered_products.id",
                        col("ordered_products.name").alias("product_name"),
                        "ordered_products.price",
                        "ordered_products.curr",
                        "ordered_products.qty",
                        "ordered_products.unit",
                    ).distinct()
        return df_parsed



#####################################
############--SCD - 1--##############
#####################################

def scd_merge_table(spark,source_table,target_table,business_key):

    print("Checking if Silver Table exists")
    if not spark.catalog.tableExists(target_table):
        print("First Load: Creating Silver Table ")
        source_table.write.format("delta").mode("overwrite").saveAsTable(target_table)

    else:
        print("Incremental Load: Performing SCD Type 1 Merge",target_table)

        delta_table = DeltaTable.forName(spark,target_table)

        merge_condition = " AND ".join(
            [f"target.{col} = source.{col}" for col in business_key]
        )

        

        delta_table.alias("target").merge(source_table.alias("source"),
                                        merge_condition)\
                                            .whenMatchedUpdateAll()\
                                                .whenNotMatchedInsertAll()\
                                                    .execute()
    print("Merge successfully")
    

##############################
#########--PROMOTIONS--#######
##############################

from pyspark.sql.types import ArrayType

def promotions(df):
    promo_info_schema = ArrayType(StructType([
        StructField("promo_disc", StringType()),
        StructField("promo_id", StringType()),
        StructField("promo_item", StringType()),
        StructField("promo_qty", StringType())
    ]))
    df_promo_parsed = df.withColumn("promo_info", from_json(col("promo_info"), promo_info_schema))\
                        .withColumn("promo_info",explode_outer("promo_info")).filter(col("promo_info").isNotNull())\
                        .select(
                            "customer_id",
                            "customer_name",
                            "order_number",
                            "promo_info.*"              
                        ).distinct()
    return df_promo_parsed


#########################
####--CLICKED ITEMS--####    
#########################

def clicked_items(df):
    
    clicked_items_schema = ArrayType(
        ArrayType(StringType())
            ) 


    clicked_item = df.withColumn("clicked_items",from_json(col("clicked_items"),clicked_items_schema))\
                .withColumn("clicked_items",explode_outer("clicked_items")).filter(col("clicked_items").isNotNull())\
                .select(
                    "customer_id",
                    "customer_name",
                    "order_number",
                    col("clicked_items")[0].alias("product_id"),
                    col("clicked_items")[1].alias("score")
                    ).distinct()
    return clicked_item



 #####################
 ####--MAIN CODE--####    
 #####################                   
 
print(f"Reading source table")
bronze_table = "ecommerce_analytics.bronze.sales_orders"
df = spark.read.table(bronze_table)

# Transformations
print("Transforming data Started")
sales_order_df = sales_orders(df)
ordered_products_df = orderd_product(df)
promotions_df = promotions(df)
clicked_items_df = clicked_items(df)

# Write

scd_merge_table(spark,sales_order_df,"ecommerce_analytics.silver.sales_orders",["order_number", "number_of_line_items"])

scd_merge_table(spark,ordered_products_df,"ecommerce_analytics.silver.orderd_products",["order_number","id","price"])

scd_merge_table(spark,promotions_df,"ecommerce_analytics.silver.promotions",["order_number","promo_id","promo_item","promo_qty","promo_disc"])

scd_merge_table(spark,clicked_items_df,"ecommerce_analytics.silver.clicked_items",["customer_id","order_number","product_id","score"])
 



print(f"successfully wrote all tables")
